# Daily Quant Trading Pipeline (paper trading)

This notebook runs the end-to-end daily pipeline and inspects signals, trades, and state. The core logic lives in `pipeline/`; this notebook is just a driver/inspector.

In [1]:
from pathlib import Path
import sys

# In notebooks, __file__ is usually undefined; fall back to CWD
project_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/saibharadwaj/Desktop/stat107/saib2/project2


In [2]:
import importlib
import subprocess
import sys

# Ensure dependencies are installed using THIS notebook's Python.
# If you ever see errors like "numpy.dtype size changed" (binary mismatch),
# re-run this cell; it force-reinstalls pinned, compatible wheels.
req_path = project_root / "requirements.txt"

print("Using Python:", sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

if not req_path.exists():
    raise FileNotFoundError(f"requirements.txt not found at {req_path}")

# Force reinstall pinned deps to avoid numpy/sklearn binary incompatibility
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--no-cache-dir",
        "-r",
        str(req_path),
    ]
)

# Sanity check imports
for m in ["numpy", "pandas", "sklearn", "yfinance"]:
    importlib.import_module(m)
print("Imports OK:")
import numpy as np
import pandas as pd
import sklearn
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)


Using Python: /Users/saibharadwaj/Desktop/stat107/saib2/project2/.venv/bin/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 8.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 8.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 9.6 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 7.4 MB/s  0:00:02 eta 0:00:01
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl siz

In [2]:
# Run the daily pipeline once (pulls data, computes signals, logs, updates state)
from pipeline.core import run_daily

run_daily()
print("Daily run completed.")

Daily run completed.


In [3]:
import pandas as pd
from pathlib import Path

logs_dir = project_root / "logs"

def tail_csv(path: Path, n: int = 5):
    if not path.exists():
        print(f"No file: {path}")
        return
    display(pd.read_csv(path, on_bad_lines="skip", engine="python").tail(n))

print("Latest signals (tail):")
tail_csv(logs_dir / "daily_signals.csv")

print("Latest trades (tail):")
tail_csv(logs_dir / "trades.csv")

print("Performance (tail):")
tail_csv(logs_dir / "performance.csv")

Latest signals (tail):


,date,symbol,action,reason,confidence,fast_sma,slow_sma,close,execution_price,rel_strength,volatility,fundamentals_passed,fundamentals_pe,fundamentals_ps,ml_prob_up,ml_model
264,2026-01-13,IBM,HOLD,no_crossover,1.0,302.246001,287.799399,303.320007,next_open,0.044163,0.238831,True,36.109540,4.335096,0.542241,xgb_classification
265,2026-01-13,SAP,HOLD,no_crossover,1.0,243.825500,257.098300,246.589996,next_open,-0.130602,0.188747,True,35.126780,7.884031,0.547878,xgb_classification
266,2026-01-13,NOW,HOLD,no_crossover,1.0,150.845099,173.040700,137.559998,next_open,-0.273015,0.479540,False,83.375760,11.459385,0.555295,xgb_classification
267,2026-01-13,PANW,HOLD,no_crossover,1.0,187.031751,199.188650,190.335007,next_open,-0.118636,0.276191,False,119.707550,13.882017,0.538230,xgb_classification
268,2026-01-13,NFLX,HOLD,no_crossover,1.0,92.556250,110.376740,89.845001,next_open,-0.285085,0.147489,True,37.439583,8.777209,0.545588,xgb_classification


Latest trades (tail):
No file: /Users/saibharadwaj/Desktop/stat107/saib2/project2/logs/trades.csv
Performance (tail):


,timestamp,date,portfolio_value,cash,open_positions,pending_orders
14,2026-01-13T20:18:58.823812,2026-01-13,1.0,1.0,0,0
15,2026-01-13T20:20:36.175141,2026-01-13,1.0,1.0,0,0
16,2026-01-13T20:22:29.507533,2026-01-13,1.0,1.0,0,0
17,2026-01-13T20:24:51.213701,2026-01-13,1.0,1.0,0,0
18,2026-01-13T20:25:37.060464,2026-01-13,1.0,1.0,0,0


In [4]:
# Inspect benchmark processed data (QQQ)
import pandas as pd
from pathlib import Path

bench_path = project_root / "data" / "processed" / "QQQ.csv"
if bench_path.exists():
    display(pd.read_csv(bench_path).head())
else:
    print(f"No benchmark file yet at {bench_path}")

,Date,"('QQQ', 'AdjClose')","('QQQ', 'AdjClose').1","('QQQ', 'AdjClose').2","('QQQ', 'Close')","('QQQ', 'Close').1","('QQQ', 'Close').2","('QQQ', 'High')","('QQQ', 'High').1","('QQQ', 'High').2",...,Close,High,Low,Open,Volume,Return,FastSMA,SlowSMA,Volatility,Valid
0,2016-05-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,109.239998,109.570000,108.680000,108.870003,37951700.0,0.007192,106.2515,105.4526,0.147776,True
1,2016-05-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,109.559998,109.699997,109.089996,109.349998,21885900.0,0.002929,106.4155,105.4532,0.139800,True
2,2016-05-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,110.129997,110.139999,109.610001,109.629997,19870000.0,0.005203,106.6360,105.4614,0.137947,True
3,2016-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,110.339996,110.510002,109.779999,110.279999,24366100.0,0.001907,106.8170,105.4822,0.135167,True
4,2016-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,110.349998,110.599998,109.910004,110.000000,19571500.0,0.000091,107.0480,105.5370,0.129024,True


In [5]:
import json
from pipeline.config import StrategyConfig
from pipeline.core import load_state

cfg = StrategyConfig()
state = load_state(cfg.state_path)
print(json.dumps(state, indent=2))

{
  "cash": 1.0,
  "last_valuation_date": "2026-01-13",
  "pending_orders": [],
  "portfolio_value": 1.0,
  "positions": {}
}


In [6]:
# Latest full signals snapshot (all symbols)
import pandas as pd
from pathlib import Path

signals_path = project_root / "logs" / "daily_signals.csv"
if not signals_path.exists():
    print(f"No signals file at {signals_path}")
else:
    df = pd.read_csv(signals_path, on_bad_lines="skip", engine="python")
    if "date" not in df.columns:
        print("Signals file missing 'date' column")
    else:
        df.loc[:, "date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])  # drop non-date rows to avoid Timestamp/float compare
        df = df.sort_values(["date", "symbol"]).drop_duplicates(
            subset=["symbol", "date"], keep="last"
        )
        if df.empty:
            print("No valid date rows in signals file")
        else:
            latest_date = df["date"].max()
            latest_df = df[df["date"] == latest_date].copy()
            latest_df = latest_df.sort_values(["symbol"]).reset_index(drop=True)

            # Rename columns to uppercase with units/meaning where helpful
            col_map = {
                "date": "DATE",
                "symbol": "SYMBOL",
                "action": "ACTION",
                "reason": "REASON",
                "confidence": "CONFIDENCE_0_1",
                "rel_strength": "REL_STRENGTH_60D_VS_QQQ",
                "volatility": "VOL_ANNUAL",
                "fast_sma": "FAST_SMA",
                "slow_sma": "SLOW_SMA",
                "close": "CLOSE",
                "execution_price": "EXEC_PRICE",
                "fundamentals_passed": "FUNDAMENTALS_OK",
                "fundamentals_pe": "PE",
                "fundamentals_ps": "PS",
            }
            latest_df = latest_df.rename(columns=col_map)

            legend = [
                "CONFIDENCE_0_1: 0-1 score from SMA spread / price & vol",
                "REL_STRENGTH_60D_VS_QQQ: 60d return minus QQQ return (fraction)",
                "VOL_ANNUAL: annualized volatility (fraction, e.g., 0.25 = 25%)",
                "SMA columns in price units; CLOSE is last close used for signal",
                "FUNDAMENTALS_OK: True if PE/PS pass configured limits",
            ]
            print("; ".join(legend))

            with pd.option_context(
                "display.max_rows", None,
                "display.max_columns", None,
                "display.float_format", "{:.4f}".format,
            ):
                display(latest_df)



CONFIDENCE_0_1: 0-1 score from SMA spread / price & vol; REL_STRENGTH_60D_VS_QQQ: 60d return minus QQQ return (fraction); VOL_ANNUAL: annualized volatility (fraction, e.g., 0.25 = 25%); SMA columns in price units; CLOSE is last close used for signal; FUNDAMENTALS_OK: True if PE/PS pass configured limits


,DATE,SYMBOL,ACTION,REASON,CONFIDENCE_0_1,FAST_SMA,SLOW_SMA,CLOSE,EXEC_PRICE,REL_STRENGTH_60D_VS_QQQ,VOL_ANNUAL,FUNDAMENTALS_OK,PE,PS,ml_prob_up,ml_model
0,2026-01-13 00:00:00,AAPL,HOLD,no_crossover,1.0000,268.7240,259.7635,259.2000,next_open,-0.0068,0.1120,True,34.7034,9.2045,0.5608,xgb_classification
1,2026-01-13 00:00:00,ADBE,HOLD,no_crossover,0.2021,344.3518,343.6680,309.6850,next_open,-0.1049,0.2859,True,18.5314,5.5264,0.5801,xgb_classification
2,2026-01-13 00:00:00,AMD,HOLD,no_crossover,1.0000,212.0667,205.8093,222.5350,next_open,-0.0794,0.4697,False,116.3927,11.3008,0.4467,xgb_classification
3,2026-01-13 00:00:00,AMZN,HOLD,no_crossover,1.0000,233.2993,229.5408,242.4250,next_open,0.1037,0.2313,True,34.2914,3.7489,0.5475,xgb_classification
4,2026-01-13 00:00:00,AVGO,HOLD,no_crossover,1.0000,343.9578,348.5148,355.3550,next_open,-0.0169,0.3674,False,74.8095,26.3714,0.5064,xgb_classification
5,2026-01-13 00:00:00,CRM,HOLD,no_crossover,1.0000,260.1525,248.7437,242.4300,next_open,-0.0369,0.3398,True,32.3173,5.7233,0.5733,xgb_classification
6,2026-01-13 00:00:00,GOOGL,HOLD,no_crossover,1.0000,315.1355,274.2288,335.5600,next_open,0.2906,0.1922,True,33.1571,10.5434,0.4893,xgb_classification
7,2026-01-13 00:00:00,IBM,HOLD,no_crossover,1.0000,302.2460,287.7994,303.3200,next_open,0.0442,0.2388,True,36.1095,4.3351,0.5422,xgb_classification
8,2026-01-13 00:00:00,INTC,HOLD,no_crossover,1.0000,38.9726,35.0906,47.3825,next_open,0.2461,0.6098,False,789.5834,4.2287,0.4285,xgb_classification
9,2026-01-13 00:00:00,META,HOLD,no_crossover,1.0000,655.3415,690.4237,627.8400,next_open,-0.1584,0.1970,True,27.7788,8.3528,0.5370,xgb_classification


In [7]:
# Simple backtest stats from logged performance/trades
import pandas as pd
import numpy as np
from pathlib import Path

perf_path = project_root / "logs" / "performance.csv"
trades_path = project_root / "logs" / "trades.csv"

if not perf_path.exists():
    print(f"No performance file at {perf_path}. Run the pipeline first.")
else:
    perf = pd.read_csv(perf_path, on_bad_lines="skip", engine="python")
    if "date" not in perf.columns or "portfolio_value" not in perf.columns:
        print("performance.csv missing required columns")
    else:
        perf["date"] = pd.to_datetime(perf["date"], errors="coerce")
        perf = perf.dropna(subset=["date", "portfolio_value"]).sort_values("date")
        perf = perf.drop_duplicates(subset="date", keep="last")
        perf = perf.reset_index(drop=True)
        if len(perf) < 2:
            print("Not enough performance points to compute stats.")
        else:
            equity = perf["portfolio_value"].astype(float)
            rets = equity.pct_change().dropna()
            if len(rets) == 0:
                print("No returns to evaluate yet.")
            else:
                n_days = len(rets)
                cagr = (equity.iloc[-1] / equity.iloc[0]) ** (252 / n_days) - 1 if equity.iloc[0] > 0 else np.nan
                sharpe = (rets.mean() * np.sqrt(252)) / rets.std(ddof=0) if rets.std(ddof=0) > 0 else np.nan
                roll_max = equity.cummax()
                dd = (equity - roll_max) / roll_max
                max_dd = dd.min()
                print(f"Observations: {n_days} trading days")
                print(f"CAGR: {cagr:.2%}" if pd.notna(cagr) else "CAGR: n/a")
                print(f"Sharpe (daily->annualized): {sharpe:.2f}" if pd.notna(sharpe) else "Sharpe: n/a")
                print(f"Max drawdown: {max_dd:.2%}" if pd.notna(max_dd) else "Max drawdown: n/a")

if trades_path.exists():
    trades = pd.read_csv(trades_path, on_bad_lines="skip", engine="python")
    if not trades.empty and "realized_pnl" in trades.columns:
        wins = trades[trades["realized_pnl"] > 0]
        win_rate = len(wins) / len(trades) if len(trades) > 0 else np.nan
        print(f"Trades: {len(trades)}, Win rate: {win_rate:.2%}" if pd.notna(win_rate) else "Trades: n/a")


Observations: 2 trading days
CAGR: 0.00%
Sharpe: n/a
Max drawdown: 0.00%


In [10]:
# Train pooled XGBoost classifier on ~10y history (one-time or periodic)
# NOTE: This requires a compatible Python environment (recommended: Python 3.11 venv)
import sys
from pipeline.config import StrategyConfig
from pipeline.ml import tune_and_train_pooled_classifier

print("PYTHON:", sys.version)

cfg = StrategyConfig()
meta = tune_and_train_pooled_classifier(cfg, trials=30, seed=7)
print("VALID_ACCURACY:", meta.get("valid_accuracy"))
print("VALID_AUC:", meta.get("valid_auc"))
print("VALID_LOGLOSS:", meta.get("valid_logloss"))
print("BEST_PARAMS:", meta.get("best_params"))
print(f"Model saved to {cfg.ml_model_path}")

PYTHON: 3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.4.4.1)]
VALID_ACCURACY: 0.5215311004784688
VALID_AUC: 0.5075747982965685
VALID_LOGLOSS: 0.6937217715300654
BEST_PARAMS: {'n_estimators': 274, 'learning_rate': 0.05844860147574698, 'max_depth': 2, 'min_child_weight': 19.01435716680548, 'subsample': 0.8293330870444484, 'colsample_bytree': 0.7360272269683359, 'reg_lambda': 5.430492396175706, 'reg_alpha': 4.760197453537611, 'gamma': 2.222391048235566}
Model saved to /Users/saibharadwaj/Desktop/stat107/saib2/project2/models/xgb_classification.json


In [8]:
# 5-minute intraday backtest (view results in-notebook)
# NOTE: Yahoo 5m data is limited (often ~60 trading days max).

import importlib
import pipeline.config as _cfg_mod

# If you edited code and your notebook kernel is still running, reload to pick up new fields.
importlib.reload(_cfg_mod)
from pipeline.config import StrategyConfig

import pipeline.intraday_backtest as _ibt
importlib.reload(_ibt)
from pipeline.intraday_backtest import run_intraday_backtest

cfg = StrategyConfig()

# Optional: shorten for faster downloads while still having enough bars for SMA warmup
cfg.intraday_period = "5d"  # try "30d" or "60d" for more history

# Optional: more "day-tradey" SMA windows (in 5m bars)
# cfg.fast_window = 12   # ~1 hour
# cfg.slow_window = 48   # ~4 hours
# cfg.vol_window = 78    # ~1 trading day

intraday_equity, intraday_trades, intraday_report = run_intraday_backtest(
    cfg,
    initial_value=1.0,
)

print("Intraday metrics:", intraday_report.get("metrics"))
print("Trades:", intraday_report.get("trades"))

# `tail()` only shows the last 5 rows; it doesn't mean only 5 symbols were used.
# Show a per-symbol summary so you can see all 19 tickers.
import pandas as pd

if intraday_trades is None or intraday_trades.empty:
    print("No trades in this window. Try a longer intraday_period (e.g. 30d/60d) or shorter SMA windows.")
else:
    intraday_trades = intraday_trades.copy()
    intraday_trades["realized_pnl"] = pd.to_numeric(intraday_trades.get("realized_pnl"), errors="coerce")

    sym_stats = (
        intraday_trades.groupby("symbol", dropna=False)
        .agg(
            trades=("action", "count"),
            buys=("action", lambda s: int((s == "BUY").sum())),
            sells=("action", lambda s: int((s == "SELL").sum())),
            realized_pnl_sum=("realized_pnl", "sum"),
        )
        .reindex(cfg.symbols, fill_value=0)
        .reset_index()
    )

    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(sym_stats)

# Equity + a larger trades slice
display(intraday_equity.tail(20))
display(intraday_trades.tail(50))

# Files are also written when you run the CLI:
#   python -m pipeline.intraday_backtest


Intraday metrics: {'observations': 283, 'total_return': -0.005471876285127575, 'cagr': -0.316888596315568, 'annual_vol': 0.11189814005090967, 'sharpe': -3.3500366287574, 'max_drawdown': -0.012299001625179128, 'bars_per_year': 19656}
Trades: {'count': 18, 'win_rate': 0.0}


,symbol,trades,buys,sells,realized_pnl_sum
0,AAPL,2,1,1,-0.001515
1,MSFT,0,0,0,0.000000
2,AMZN,0,0,0,0.000000
3,GOOGL,0,0,0,0.000000
4,META,0,0,0,0.000000
5,NVDA,4,2,2,-0.003885
6,AVGO,0,0,0,0.000000
7,AMD,1,1,0,0.000000
8,CRM,2,1,1,-0.001388
9,ORCL,0,0,0,0.000000


,timestamp,portfolio_value,cash,open_positions,pending_orders
264,2026-01-13T13:45:00-05:00,0.995134,0.399681,4,0
265,2026-01-13T13:50:00-05:00,0.994732,0.399681,4,0
266,2026-01-13T13:55:00-05:00,0.994857,0.399681,4,0
267,2026-01-13T14:00:00-05:00,0.994704,0.399681,4,0
268,2026-01-13T14:05:00-05:00,0.994410,0.399681,4,1
269,2026-01-13T14:10:00-05:00,0.994412,0.698095,3,0
270,2026-01-13T14:15:00-05:00,0.995009,0.698095,3,0
271,2026-01-13T14:20:00-05:00,0.994876,0.698095,3,0
272,2026-01-13T14:25:00-05:00,0.995094,0.698095,3,0
273,2026-01-13T14:30:00-05:00,0.994729,0.698095,3,0


,date,symbol,action,price,price_raw,slippage_bps,units,notional,commission,notional_plus_commission,decision_date,reason,proceeds,realized_pnl
0,2026-01-12T09:55:00-05:00,NFLX,BUY,90.174017,90.165001,1.0,0.001824,0.164443,8.222127e-06,0.164451,2026-01-12T09:50:00-05:00,fast_cross_above_slow,NaN,NaN
1,2026-01-12T10:30:00-05:00,AMD,BUY,208.855890,208.835007,1.0,0.000541,0.112929,5.646460e-06,0.112935,2026-01-12T10:25:00-05:00,fast_cross_above_slow,NaN,NaN
2,2026-01-12T11:30:00-05:00,CRM,BUY,261.126116,261.100006,1.0,0.000858,0.223952,1.119758e-05,0.223963,2026-01-12T11:25:00-05:00,fast_cross_above_slow,NaN,NaN
3,2026-01-12T11:30:00-05:00,NOW,BUY,143.159319,143.145004,1.0,0.001371,0.196290,9.814481e-06,0.196299,2026-01-12T11:25:00-05:00,fast_cross_above_slow,NaN,NaN
4,2026-01-12T11:35:00-05:00,PANW,BUY,189.348935,189.330002,1.0,0.001583,0.299710,1.498552e-05,0.299725,2026-01-12T11:30:00-05:00,fast_cross_above_slow,NaN,NaN
5,2026-01-12T12:35:00-05:00,NVDA,BUY,186.268518,186.249893,1.0,0.000014,0.002627,1.313321e-07,0.002627,2026-01-12T12:30:00-05:00,fast_cross_above_slow,NaN,NaN
6,2026-01-12T13:15:00-05:00,PANW,SELL,189.286062,189.304993,1.0,0.001583,0.299611,1.498055e-05,NaN,2026-01-12T13:10:00-05:00,fast_cross_below_slow,0.299596,-0.000129
7,2026-01-12T15:50:00-05:00,CRM,SELL,259.534042,259.559998,1.0,0.000858,0.222586,1.112931e-05,NaN,2026-01-12T15:45:00-05:00,fast_cross_below_slow,0.222575,-0.001388
8,2026-01-13T09:50:00-05:00,PANW,BUY,191.709171,191.690002,1.0,0.000665,0.127418,6.370884e-06,0.127424,2026-01-13T09:45:00-05:00,fast_cross_above_slow,NaN,NaN
9,2026-01-13T09:55:00-05:00,NVDA,SELL,184.407964,184.426407,1.0,0.000014,0.002600,1.300203e-07,NaN,2026-01-13T09:50:00-05:00,fast_cross_below_slow,0.002600,-0.000026
